# Pydantic Field Validators: Custom Input Rules

Standard type annotations and constraints handle most common requirements, but sometimes fields need custom checks or formatting. Pydantic provides the `@field_validator` decorator to run custom logic on individual fields.

In this notebook, we cover:
1. Declaring field validators using `@field_validator`.
2. Validator modes: `before` vs `after` (default).
3. Preprocessing/mutating input values (e.g. converting names to uppercase) and raising custom `ValueError` exceptions.


In [10]:
!uv pip install pydantic 'pydantic[email]' --quiet


## 1. Imports

We import standard classes along with the `@field_validator` decorator.


In [11]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated


## 2. Defining a Model with Field Validators

Validators in Pydantic:
*   Must be decorated with `@field_validator('<field_name>')`.
*   Must be declared as `@classmethod`.
*   Receive the value of the field as the input argument.
*   Must return the validated/transformed value or raise a `ValueError` (or `AssertionError`).

### Validator Modes
1.  **`after` mode (Default)**: Runs *after* standard Pydantic validation. The validator receives a value that has already been coerced to the correct type (e.g., standard float constraints, email syntax verification).
2.  **`before` mode**: Runs *before* standard Pydantic validation. The validator receives the raw, uncoerced input value. This is ideal for preprocessing (e.g., parsing dates from complex string formats, sanitizing spaces).


In [12]:
class Patient(BaseModel):

    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    age: int
    linkedin_url: Optional[AnyUrl] = None
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: Dict[str, str]

    # Mode 'after' validator (default): runs after name is confirmed as string.
    # It sanitizes and transforms the name to uppercase.
    @field_validator('name')
    @classmethod
    def transform_name(cls, value: str) -> str:
        print(f"[Validator: name] Transforming input: '{value}'")
        return value.upper()
    
    # Mode 'before' validator: runs before standard Pydantic parsing.
    # It receives the raw input value. We validate that age fits our domain bounds.
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value: any) -> int:
        print(f"[Validator: age] Validating raw age input: {value}")
        if isinstance(value, int) and 0 < value < 120:
            return value
        else:
            raise ValueError("Age should be in between 0 and 120")


## 3. Helper Functions and Input Data

We define our mock data flows and create a patient input dictionary.


In [13]:
def insert_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info inserted')


In [14]:
def update_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info updateed')


In [15]:
patient_info = {
    'name': 'Kevin',
    'age': 165,
    'weight': 67.9,
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': {
        'email_id': 'example@domain.io',
        'contact_number': '9999999999'
    }
}


## 4. Parsing Data & Verification

Instantiate the model. Note that the console output shows the order of validators, and the printed model instance shows `name='KEVIN'` (fully transformed to uppercase).


In [16]:
patient = Patient(**patient_info)
patient


[Validator: name] Transforming input: 'Kevin'
[Validator: age] Validating raw age input: 165


ValidationError: 1 validation error for Patient
age
  Value error, Age should be in between 0 and 120 [type=value_error, input_value=165, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [ ]:
insert_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 65
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: {'email_id': 'example@domain.io', 'contact_number': '9999999999'}
Patient info inserted


In [ ]:
update_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 65
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: {'email_id': 'example@domain.io', 'contact_number': '9999999999'}
Patient info updateed
